# KROM v2 — Flow Past a Cylinder (Re = 100)

**Method:** Kernel Reduced Order Model with two new features over the baseline:
1. **Registered empirical kernel** — each training snapshot is phase-aligned via
   an FFT x-shift before building the empirical covariance, making the kernel
   invariant to the dominant streamwise translation symmetry.
2. **Time-local sparse Cholesky factors** — one sparse Cholesky factor is precomputed
   offline per KROM time step from the registered snapshots at that phase of the
   shedding cycle. Online rollout only performs triangular solves.

**Problem:** 2-D incompressible NS, vorticity–streamfunction formulation, cylinder
enforced via Brinkman volume penalization on a doubly-periodic domain [0,8]×[0,4].

**Comparison:** unregistered time-pooled KROM v1 vs registered time-local KROM v2
vs FOM coarse-grid reference, on unseen initial conditions.

In [ ]:
from __future__ import annotations
import sys, time, math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import welch

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.krom.cylinder_flow import (
    CylinderNSConfig, build_fom_grid, build_krom_grid,
    generate_cylinder_dataset, fom_rollout_at_krom_resolution, compute_drag_lift,
)
from src.krom.registration import (
    register_snapshot_bank, register_test_snapshot, deregister_snapshot, compute_x_shift,
)
from src.krom.sparse_cholesky import make_snapshot_kernel_source, sparse_precision_factor
from src.krom.ordering import build_measurement_ordering
from src.krom.gauss_newton import solve_gauss_newton, ResidualOperator

torch.set_default_dtype(torch.float64)
print('Imports OK')

## 1. Configuration

In [ ]:
cfg = CylinderNSConfig(
    Lx=8.0, Ly=4.0,
    Nx=256, Ny=128,          # FOM pseudo-spectral resolution
    xc=2.0, yc=2.0, R=0.25, # cylinder: D = 0.5, blockage ≈ 12.5%
    U_bg=1.0, Re=100.0,
    epsilon=1e-3,             # Brinkman penalty
    dt_fom=0.005,
    Nx_krom=64, Ny_krom=32,  # KROM collocation grid (2048 points)
    dt_krom=0.1,
)
print(f'nu = {cfg.nu:.4f},  D = {cfg.D},  Re = {cfg.Re}')
print(f'FOM grid  : {cfg.Nx}x{cfg.Ny}')
print(f'KROM grid : {cfg.Nx_krom}x{cfg.Ny_krom} = {cfg.Nx_krom*cfg.Ny_krom} pts')
print(f'FOM steps per KROM step: {cfg.krom_steps_per_save}')

# Experimental parameters
M_TRAIN  = 20    # training trajectories
M_TEST   = 5     # test (unseen) trajectories
T_WARMUP = 15.0  # warm-up to reach attractor
T_TRAJ   = 6.0   # trajectory length for data collection
RHO      = 4.0   # sparse Cholesky radius
NUGGET   = 1e-9
GN_STEPS = 3     # Gauss-Newton iterations per KROM step
GN_DAMP  = 1e-8

T_SNAP = round(T_TRAJ / cfg.dt_krom)  # KROM steps per trajectory
N_KROM = cfg.Nx_krom * cfg.Ny_krom
print(f'KROM steps per trajectory: {T_SNAP}  (state dim: {3*N_KROM})')

## 2. Build Grids

In [ ]:
grid_fom  = build_fom_grid(cfg)
grid_krom = build_krom_grid(cfg)
krom_points = grid_krom['points']  # (N_KROM, 2) torch tensor

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].contourf(grid_fom['X'],  grid_fom['Y'],  grid_fom['chi'],  20, cmap='Blues')
axes[0].set_title('FOM cylinder mask chi (256x128)'); axes[0].set_aspect('equal')
axes[1].contourf(grid_krom['X'], grid_krom['Y'], grid_krom['chi'], 20, cmap='Blues')
axes[1].set_title('KROM cylinder mask chi (64x32)'); axes[1].set_aspect('equal')
plt.tight_layout(); plt.savefig('fig_cylinder_masks.png', dpi=100); plt.show()

## 3. Generate Training and Test Datasets

In [ ]:
print(f'Generating {M_TRAIN} training trajectories  (T_warmup={T_WARMUP}, T_traj={T_TRAJ})...')
t0 = time.perf_counter()
train_data = generate_cylinder_dataset(
    cfg, grid_fom, n_traj=M_TRAIN, T_warmup=T_WARMUP, T_traj=T_TRAJ,
    seeds=list(range(M_TRAIN)),
)
print(f'  Done in {time.perf_counter()-t0:.1f}s.  omega shape: {train_data["omega"].shape}')

print(f'Generating {M_TEST} test trajectories...')
t0 = time.perf_counter()
test_data = generate_cylinder_dataset(
    cfg, grid_fom, n_traj=M_TEST, T_warmup=T_WARMUP, T_traj=T_TRAJ,
    seeds=list(range(1000, 1000 + M_TEST)),
)
print(f'  Done in {time.perf_counter()-t0:.1f}s.')

In [ ]:
# Quick check: vorticity snapshots at t=0, T/2, T for training IC 0
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, t_idx in zip(axes, [0, T_SNAP//2, T_SNAP]):
    om = train_data['omega'][0, t_idx]
    vmax = max(abs(om.min()), abs(om.max()))
    ax.contourf(grid_krom['X'], grid_krom['Y'], om, 40, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.contour(grid_krom['X'], grid_krom['Y'], grid_krom['chi'], [0.5], colors='k', linewidths=1.5)
    ax.set_title(f't = {t_idx * cfg.dt_krom:.1f}'); ax.set_aspect('equal')
plt.suptitle('Training IC 0 — vorticity at KROM grid')
plt.tight_layout(); plt.savefig('fig_training_snapshots.png', dpi=100); plt.show()

## 4. FFT Registration of Training Snapshots

In [ ]:
KX_k = grid_krom['KX']   # (Nx_k, Ny_k)
KY_k = grid_krom['KY']
K2_k = grid_krom['K2']
kx_k = KX_k[:, 0]        # 1-D x-wavenumber array (Nx_k,)

print('Registering training snapshots (FFT x-shift, per trajectory per time step)...')
t0 = time.perf_counter()
reg_train = register_snapshot_bank(
    train_data['omega'],   # (M, T+1, Nx_k, Ny_k)
    Lx=cfg.Lx, kx=kx_k,
    KX=KX_k, KY=KY_k, K2=K2_k,
    ref_traj=0,
)
print(f'Done in {time.perf_counter()-t0:.1f}s.  '
      f'Shift range: [{reg_train["shifts"].min():.3f}, {reg_train["shifts"].max():.3f}]')

# Show the registration effect for 4 trajectories at t = T/2
t_show = T_SNAP // 2
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for col in range(4):
    raw = train_data['omega'][col, t_show]
    reg = reg_train['omega'][col, t_show]
    vmax = max(abs(raw).max(), abs(reg).max())
    axes[0, col].contourf(grid_krom['X'], grid_krom['Y'], raw, 30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0, col].set_title(f'Raw IC {col}'); axes[0, col].set_aspect('equal')
    axes[1, col].contourf(grid_krom['X'], grid_krom['Y'], reg, 30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f'Registered IC {col}'); axes[1, col].set_aspect('equal')
axes[0, 0].set_ylabel('Raw'); axes[1, 0].set_ylabel('Registered')
plt.suptitle(f'Registration effect at t = {t_show * cfg.dt_krom:.1f}')
plt.tight_layout(); plt.savefig('fig_registration.png', dpi=100); plt.show()

In [ ]:
# Singular values before vs after registration: registration should collapse rank
t_check = T_SNAP // 2
raw_mat = train_data['omega'][:, t_check].reshape(M_TRAIN, -1)
reg_mat = reg_train['omega'][:, t_check].reshape(M_TRAIN, -1)
sv_raw = np.linalg.svd(raw_mat, compute_uv=False)
sv_reg = np.linalg.svd(reg_mat, compute_uv=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(sv_raw / sv_raw[0], 'o-', label='Unregistered', ms=4)
ax.semilogy(sv_reg / sv_reg[0], 's-', label='Registered',   ms=4)
ax.set_xlabel('Index'); ax.set_ylabel('Normalised singular value')
ax.set_title(f'Rank decay at t = {t_check * cfg.dt_krom:.1f}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('fig_singular_values.png', dpi=100); plt.show()
top3_raw = (sv_raw[:3]**2).sum() / (sv_raw**2).sum()
top3_reg = (sv_reg[:3]**2).sum() / (sv_reg**2).sum()
print(f'Energy in top-3 singular values  raw: {top3_raw:.3f}   registered: {top3_reg:.3f}')

## 5. Measurement Ordering (built once from the KROM spatial grid)

In [ ]:
# Three derivative groups (dx, dy, lap): each covers all N_KROM spatial points
all_idx = torch.arange(N_KROM, dtype=torch.long)
deriv_groups = (all_idx, all_idx, all_idx)

print(f'Building maximin ordering for {N_KROM} spatial + 3x{N_KROM} derivative points...')
t0 = time.perf_counter()
ordering = build_measurement_ordering(krom_points, deriv_groups, k_neighbors=3)
print(f'Done in {time.perf_counter()-t0:.2f}s')

## 6. Precompute Sparse Cholesky Factors (Offline)

In [ ]:
def build_factor_at_timestep(t_idx, reg, krom_points, deriv_groups, ordering, rho, nugget):
    """Build registered SnapshotKernelSource from M training snapshots at step t_idx,
    then compute and return the sparse Cholesky precision factor."""
    M = reg['omega'].shape[0]
    N = krom_points.shape[0]
    scale = 1.0 / M

    def _feat(arr):
        # arr: (M, T+1, Nx_k, Ny_k) → slice at t_idx → (N, M)
        return torch.from_numpy(arr[:, t_idx].reshape(M, N).T.copy())

    ks = make_snapshot_kernel_source(
        (_feat(reg['omega']), _feat(reg['dx']), _feat(reg['dy']), _feat(reg['lap'])),
        nugget=nugget, scale=scale,
    )
    factor, _ = sparse_precision_factor(
        theta=ks, dirac_points=krom_points, derivative_point_groups=deriv_groups,
        rho=rho, nugget=nugget, ordering=ordering,
    )
    return factor


# KROM v2: one factor per time step (T_SNAP + 1 total — step 0 seeds the IC)
print(f'Precomputing {T_SNAP + 1} KROM v2 factors (registered, time-local)...')
t_start = time.perf_counter()
factors_v2 = []
for t in range(T_SNAP + 1):
    factors_v2.append(build_factor_at_timestep(t, reg_train, krom_points, deriv_groups, ordering, RHO, NUGGET))
    if (t + 1) % 10 == 0 or t == T_SNAP:
        elapsed = time.perf_counter() - t_start
        eta = elapsed / (t + 1) * (T_SNAP - t)
        print(f'  [{t+1}/{T_SNAP+1}]  {elapsed:.0f}s elapsed  ~{eta:.0f}s remaining')
print(f'KROM v2 factors done in {time.perf_counter()-t_start:.1f}s')

In [ ]:
# KROM v1 baseline: pool ALL snapshots across all times and trajectories into one factor
print('Building KROM v1 factor (unregistered, time-pooled)...')
M_tot = M_TRAIN * (T_SNAP + 1)

def _pool_feat(arr):
    M, T1, Nx, Ny = arr.shape
    return torch.from_numpy(arr.reshape(M * T1, Nx * Ny).T.copy())

ks_v1 = make_snapshot_kernel_source(
    (_pool_feat(train_data['omega']), _pool_feat(train_data['dx']),
     _pool_feat(train_data['dy']),   _pool_feat(train_data['lap'])),
    nugget=NUGGET, scale=1.0 / M_tot,
)
t0 = time.perf_counter()
factor_v1, _ = sparse_precision_factor(
    theta=ks_v1, dirac_points=krom_points, derivative_point_groups=deriv_groups,
    rho=RHO, nugget=NUGGET, ordering=ordering,
)
print(f'KROM v1 factor done in {time.perf_counter()-t0:.2f}s')

## 7. KROM Rollout (with Brinkman Penalization)

In [ ]:
from src.krom.gauss_newton import ResidualOperator

def cylinder_vorticity_residual_operator(
    state: torch.Tensor,
    prev_w: torch.Tensor, prev_wx: torch.Tensor,
    prev_wy: torch.Tensor, prev_lap: torch.Tensor,
    vel_u: torch.Tensor, vel_v: torch.Tensor,
    chi: torch.Tensor,
    nu: float, dt: float, epsilon: float,
) -> ResidualOperator:
    """Matrix-free (JVP/VJP) residual for the penalized vorticity equation.

    The Brinkman penalization adds (1/(eps*nu)) * chi to the coeff_w term:
      lap = (2/nu) * [(w - w_p)/dt + 0.5*u*(wx+wx_p) + 0.5*v*(wy+wy_p)
                      + (1/(2*eps))*chi*(w+w_p)] - lap_p
    """
    n = prev_w.numel()
    w  = state[:n]
    wx = state[n:2*n]
    wy = state[2*n:3*n]

    # Jacobian diagonal blocks (all elementwise)
    pen_coeff = 1.0 / (nu * epsilon)  # = 1/(nu*eps) per chi element
    coeff_w  = (2.0 / (nu * dt)) * torch.ones(n, dtype=state.dtype) + pen_coeff * chi
    coeff_wx = (1.0 / nu) * vel_u
    coeff_wy = (1.0 / nu) * vel_v

    pen_term = (1.0 / (2.0 * epsilon)) * chi * (w + prev_w)
    lap = (2.0 / nu) * (
        (w - prev_w) / dt
        + 0.5 * (vel_u * (wx + prev_wx) + vel_v * (wy + prev_wy))
        + pen_term
    ) - prev_lap

    residual = torch.cat([w, wx, wy, lap], dim=0)

    def jvp(v: torch.Tensor) -> torch.Tensor:
        dw  = v[:n]
        dwx = v[n:2*n]
        dwy = v[2*n:3*n]
        dlap = coeff_w * dw + coeff_wx * dwx + coeff_wy * dwy
        return torch.cat([dw, dwx, dwy, dlap], dim=0)

    def vjp(v: torch.Tensor) -> torch.Tensor:
        vw   = v[:n]
        vwx  = v[n:2*n]
        vwy  = v[2*n:3*n]
        vlap = v[3*n:]
        return torch.cat([
            vw  + coeff_w  * vlap,
            vwx + coeff_wx * vlap,
            vwy + coeff_wy * vlap,
        ], dim=0)

    diag = torch.cat([1.0 + coeff_w**2, 1.0 + coeff_wx**2, 1.0 + coeff_wy**2], dim=0)
    return ResidualOperator(residual=residual, jvp=jvp, vjp=vjp, preconditioner_diag=2.0 * diag)


def solve_streamfunction_krom(omega_2d, grid_krom, cfg):
    """FFT Poisson solve for streamfunction psi from vorticity, on KROM grid.
    Returns u = d psi/dy + U_bg,  v = -d psi/dx."""
    KX, KY, K2 = grid_krom['KX'], grid_krom['KY'], grid_krom['K2']
    om_hat = np.fft.fft2(omega_2d)
    psi_hat = np.where(K2 == 0, 0.0, -om_hat / K2)
    u = np.fft.ifft2(1j * KY * psi_hat).real + cfg.U_bg
    v = np.fft.ifft2(-1j * KX * psi_hat).real
    return u, v


def rollout_krom(omega0_2d, factors, cfg, grid_krom, chi_t, gn_steps=2, gn_damp=1e-8):
    """Roll out KROM for T = len(factors)-1 steps starting from omega0_2d.

    ``factors`` is a list of length T+1 where factors[t] is used at step t.
    Returns vorticity array of shape (T+1, Nx_k, Ny_k).
    """
    T = len(factors) - 1
    Nx, Ny = cfg.Nx_krom, cfg.Ny_krom
    N = Nx * Ny
    nu = cfg.nu
    dt = cfg.dt_krom
    eps = cfg.epsilon

    # Initialise state from the given IC via spectral differentiation
    KX, KY, K2 = grid_krom['KX'], grid_krom['KY'], grid_krom['K2']
    om_hat0 = np.fft.fft2(omega0_2d)
    wx0  = np.fft.ifft2(1j * KX * om_hat0).real.reshape(-1)
    wy0  = np.fft.ifft2(1j * KY * om_hat0).real.reshape(-1)
    lap0 = np.fft.ifft2(-K2 * om_hat0).real.reshape(-1)

    prev_w   = torch.from_numpy(omega0_2d.reshape(-1))
    prev_wx  = torch.from_numpy(wx0)
    prev_wy  = torch.from_numpy(wy0)
    prev_lap = torch.from_numpy(lap0)
    state    = torch.cat([prev_w, prev_wx, prev_wy])

    snaps = np.zeros((T + 1, Nx, Ny))
    snaps[0] = omega0_2d.copy()

    for step in range(T):
        factor = factors[step]   # precomputed precision factor for this step

        # Velocity field from prev vorticity (explicit for stability)
        u2d, v2d = solve_streamfunction_krom(prev_w.numpy().reshape(Nx, Ny), grid_krom, cfg)
        vel_u = torch.from_numpy(u2d.reshape(-1))
        vel_v = torch.from_numpy(v2d.reshape(-1))

        _pw, _pwx, _pwy, _plap = prev_w, prev_wx, prev_wy, prev_lap

        result = solve_gauss_newton(
            initial_state=state,
            residual_and_jacobian=lambda z, _pw=_pw, _pwx=_pwx, _pwy=_pwy, _plap=_plap,
                                          vel_u=vel_u, vel_v=vel_v: 
                cylinder_vorticity_residual_operator(
                    z, _pw, _pwx, _pwy, _plap,
                    vel_u, vel_v, chi_t,
                    nu, dt, eps,
                ),
            factor=factor,
            max_iter=gn_steps,
            damping=gn_damp,
            record_history=False,
            linear_solver='cg',
            preconditioner='jacobi',
        )
        state = result.state
        new_w  = state[:N]
        new_wx = state[N:2*N]
        new_wy = state[2*N:3*N]

        # Recompute lap from the penalized equation for consistent warm-starting
        pen  = (1.0 / (2.0 * eps)) * chi_t * (new_w + _pw)
        new_lap = (2.0 / nu) * (
            (new_w - _pw) / dt
            + 0.5 * (vel_u * (new_wx + _pwx) + vel_v * (new_wy + _pwy))
            + pen
        ) - _plap

        snaps[step + 1] = new_w.detach().numpy().reshape(Nx, Ny)
        prev_w, prev_wx, prev_wy, prev_lap = new_w, new_wx, new_wy, new_lap

    return snaps


# KROM-grid cylinder mask as flat torch tensor
chi_torch = torch.from_numpy(grid_krom['chi'].reshape(-1))
print('Rollout kernel defined.')

## 8. Evaluate on Test Initial Conditions

In [ ]:
results_v2  = []   # KROM v2 (registered, time-local)
results_v1  = []   # KROM v1 (pooled, unregistered)
results_fom = []   # FOM coarse-grid reference
tau0_list   = []   # initial registration shifts (for de-registration)

omega_ref0 = train_data['omega'][0, 0]  # reference snapshot for IC registration
factors_v1_all = [factor_v1] * (T_SNAP + 1)  # single factor reused at every step

print(f'Evaluating on {M_TEST} test ICs...')
for i in range(M_TEST):
    print(f'  IC {i+1}/{M_TEST}:', end=' ', flush=True)
    omega0 = test_data['omega'][i, 0]   # (Nx_k, Ny_k)

    # --- FOM reference (coarse pseudo-spectral) ---
    t0 = time.perf_counter()
    fom_ref = fom_rollout_at_krom_resolution(omega0, T_TRAJ, cfg, grid_krom)
    t_fom = time.perf_counter() - t0
    results_fom.append(fom_ref)  # (T+1, Nx_k, Ny_k)

    # --- KROM v1 ---
    t0 = time.perf_counter()
    roll_v1 = rollout_krom(omega0, factors_v1_all, cfg, grid_krom, chi_torch,
                           gn_steps=GN_STEPS, gn_damp=GN_DAMP)
    t_v1 = time.perf_counter() - t0
    results_v1.append(roll_v1)

    # --- KROM v2 (register IC, rollout in registered space, de-register) ---
    t0 = time.perf_counter()
    omega0_reg, tau0 = register_test_snapshot(
        omega0, omega_ref0, cfg.Lx, kx_k, KX_k, KY_k, K2_k
    )
    tau0_list.append(tau0)
    roll_reg = rollout_krom(omega0_reg, factors_v2, cfg, grid_krom, chi_torch,
                            gn_steps=GN_STEPS, gn_damp=GN_DAMP)
    # De-register: undo the initial shift (apply -tau0 to all snapshots)
    # Note: for phase-exact comparison, we also look for the best shift at each t
    roll_v2_phys = np.stack([
        deregister_snapshot(roll_reg[t], tau0, kx_k)
        for t in range(T_SNAP + 1)
    ])
    t_v2 = time.perf_counter() - t0
    results_v2.append(roll_v2_phys)

    print(f'FOM {t_fom:.1f}s | v1 {t_v1:.1f}s | v2 {t_v2:.1f}s')

print('All test ICs done.')

## 9. Accuracy Metrics

In [ ]:
def relative_l2(pred, truth):
    return np.linalg.norm(pred - truth) / max(np.linalg.norm(truth), 1e-15)


times = np.arange(T_SNAP + 1) * cfg.dt_krom
err_v2 = np.zeros((M_TEST, T_SNAP + 1))
err_v1 = np.zeros((M_TEST, T_SNAP + 1))

# Phase-insensitive L2: re-align prediction to FOM at each step before measuring
for i in range(M_TEST):
    ref = results_fom[i]
    for t in range(T_SNAP + 1):
        # Find best x-shift to align prediction to reference
        tau_v2 = compute_x_shift(results_v2[i][t], ref[t], cfg.Lx)
        pred_v2_aligned = deregister_snapshot(results_v2[i][t], -tau_v2, kx_k)
        tau_v1 = compute_x_shift(results_v1[i][t], ref[t], cfg.Lx)
        pred_v1_aligned = deregister_snapshot(results_v1[i][t], -tau_v1, kx_k)
        err_v2[i, t] = relative_l2(pred_v2_aligned, ref[t])
        err_v1[i, t] = relative_l2(pred_v1_aligned, ref[t])

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(times, err_v2.mean(0), 'b-',  lw=2, label='KROM v2 (registered, time-local)')
ax.fill_between(times, err_v2.min(0), err_v2.max(0), alpha=0.2, color='b')
ax.semilogy(times, err_v1.mean(0), 'r--', lw=2, label='KROM v1 (pooled, unregistered)')
ax.fill_between(times, err_v1.min(0), err_v1.max(0), alpha=0.2, color='r')
ax.set_xlabel('Time'); ax.set_ylabel('Rel L2 error (omega, phase-aligned)')
ax.set_title(f'Re={cfg.Re},  M_train={M_TRAIN},  rho={RHO}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('fig_error_vs_time.png', dpi=100); plt.show()
print(f'Mean final rel-L2  v2={err_v2[:,-1].mean():.4f}   v1={err_v1[:,-1].mean():.4f}')

In [ ]:
# Strouhal number from lift force oscillation
def compute_strouhal(omega_traj, grid, cfg):
    """Estimate Strouhal number from lift PSD peak."""
    lifts = [compute_drag_lift(omega_traj[t], grid, cfg)[1] for t in range(len(omega_traj))]
    lifts = np.array(lifts)
    fs = 1.0 / cfg.dt_krom
    freqs, psd = welch(lifts, fs=fs, nperseg=min(64, len(lifts)//2))
    f_peak = freqs[1 + np.argmax(psd[1:])]   # skip DC
    return f_peak * cfg.D / cfg.U_bg, lifts, freqs, psd


St_fom, St_v2, St_v1 = [], [], []
print(f'{"IC":>4} | {"St_FOM":>8} | {"St_v2":>8} | {"St_v1":>8} | {"Err_v2":>8} | {"Err_v1":>8}')
print('-' * 55)
for i in range(M_TEST):
    s_fom = compute_strouhal(results_fom[i], grid_krom, cfg)[0]
    s_v2  = compute_strouhal(results_v2[i],  grid_krom, cfg)[0]
    s_v1  = compute_strouhal(results_v1[i],  grid_krom, cfg)[0]
    St_fom.append(s_fom); St_v2.append(s_v2); St_v1.append(s_v1)
    print(f'{i:>4} | {s_fom:>8.4f} | {s_v2:>8.4f} | {s_v1:>8.4f} | '
          f'{abs(s_v2-s_fom):>8.4f} | {abs(s_v1-s_fom):>8.4f}')
print('-' * 55)
print(f'{"mean":>4} | {np.mean(St_fom):>8.4f} | {np.mean(St_v2):>8.4f} | {np.mean(St_v1):>8.4f} | '
      f'{np.mean(np.abs(np.array(St_v2)-np.array(St_fom))):>8.4f} | '
      f'{np.mean(np.abs(np.array(St_v1)-np.array(St_fom))):>8.4f}')
print('\nReference: St(Re=100) ≈ 0.17–0.20')

In [ ]:
# Lift time series and PSD for first test IC
i = 0
s_fom, lift_fom, f_fom, p_fom = compute_strouhal(results_fom[i], grid_krom, cfg)
s_v2,  lift_v2,  f_v2,  p_v2  = compute_strouhal(results_v2[i],  grid_krom, cfg)
s_v1,  lift_v1,  f_v1,  p_v1  = compute_strouhal(results_v1[i],  grid_krom, cfg)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, lift_fom, 'k-',  lw=1.5, label=f'FOM  St={s_fom:.3f}')
axes[0].plot(times, lift_v2,  'b-',  lw=1.5, label=f'KROM v2  St={s_v2:.3f}')
axes[0].plot(times, lift_v1,  'r--', lw=1.5, label=f'KROM v1  St={s_v1:.3f}')
axes[0].set_xlabel('Time'); axes[0].set_ylabel('C_L')
axes[0].set_title('Lift coefficient'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(f_fom[1:], p_fom[1:], 'k-',  lw=1.5, label='FOM')
axes[1].semilogy(f_v2[1:],  p_v2[1:],  'b-',  lw=1.5, label='KROM v2')
axes[1].semilogy(f_v1[1:],  p_v1[1:],  'r--', lw=1.5, label='KROM v1')
axes[1].set_xlabel('Frequency'); axes[1].set_ylabel('PSD')
axes[1].set_title('Lift PSD'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Test IC 0 — lift dynamics')
plt.tight_layout(); plt.savefig('fig_lift_coefficients.png', dpi=100); plt.show()

In [ ]:
# Kinetic energy spectra at final time
def energy_spectrum_1d(omega_2d, Lx, Ly):
    Nx, Ny = omega_2d.shape
    om_hat = np.fft.fft2(omega_2d) / (Nx * Ny)
    kx_ = 2*np.pi/Lx * np.fft.fftfreq(Nx, 1.0/Nx)
    ky_ = 2*np.pi/Ly * np.fft.fftfreq(Ny, 1.0/Ny)
    KX_, KY_ = np.meshgrid(kx_, ky_, indexing='ij')
    K_ = np.sqrt(KX_**2 + KY_**2)
    K_[0, 0] = 1.0  # avoid division by zero
    E_2d = 0.5 * np.abs(om_hat)**2 / K_**2
    E_2d[0, 0] = 0.0
    k_max = min(Nx, Ny) // 2
    k_bins = np.arange(0.5, k_max)
    E = np.array([(E_2d[(K_ >= k-0.5) & (K_ < k+0.5)]).sum() for k in k_bins])
    return k_bins, E


fig, ax = plt.subplots(figsize=(7, 4))
for i in range(M_TEST):
    k, E_f = energy_spectrum_1d(results_fom[i][-1], cfg.Lx, cfg.Ly)
    k, E2  = energy_spectrum_1d(results_v2[i][-1],  cfg.Lx, cfg.Ly)
    k, E1  = energy_spectrum_1d(results_v1[i][-1],  cfg.Lx, cfg.Ly)
    ax.loglog(k, E_f, 'k-',  alpha=0.4, lw=1,   label='FOM' if i==0 else None)
    ax.loglog(k, E2,  'b-',  alpha=0.7, lw=1.5, label='KROM v2' if i==0 else None)
    ax.loglog(k, E1,  'r--', alpha=0.7, lw=1.5, label='KROM v1' if i==0 else None)
ax.set_xlabel('Wavenumber k'); ax.set_ylabel('E(k)')
ax.set_title(f'Energy spectra at t = {T_TRAJ:.1f}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('fig_energy_spectra.png', dpi=100); plt.show()

## 10. Vorticity Snapshot Comparison

In [ ]:
i = 0
t_indices = [T_SNAP//4, T_SNAP//2, 3*T_SNAP//4, T_SNAP]
row_labels = ['FOM coarse', 'KROM v2', 'KROM v1', '|Error| v2']

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.05, wspace=0.05)

for col, t_idx in enumerate(t_indices):
    fom = results_fom[i][t_idx]
    v2  = results_v2[i][t_idx]
    v1  = results_v1[i][t_idx]
    err = np.abs(v2 - fom)
    vmax = max(abs(fom).max(), abs(v2).max(), abs(v1).max())
    emax = max(err.max(), 1e-6)
    for row, (field, cmap, vlo, vhi) in enumerate([
        (fom, 'RdBu_r', -vmax, vmax),
        (v2,  'RdBu_r', -vmax, vmax),
        (v1,  'RdBu_r', -vmax, vmax),
        (err, 'hot',     0,    emax),
    ]):
        ax = fig.add_subplot(gs[row, col])
        ax.contourf(grid_krom['X'], grid_krom['Y'], field, 30, cmap=cmap, vmin=vlo, vmax=vhi)
        ax.contour(grid_krom['X'], grid_krom['Y'], grid_krom['chi'], [0.5], colors='k', linewidths=1)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        if col == 0: ax.set_ylabel(row_labels[row], fontsize=9)
        if row == 0: ax.set_title(f't={t_idx*cfg.dt_krom:.1f}', fontsize=9)

plt.suptitle('Vorticity comparison — test IC 0', fontsize=11)
plt.savefig('fig_vorticity_comparison.png', dpi=100, bbox_inches='tight'); plt.show()

## 11. Summary Table

In [ ]:
err_St_v2 = np.abs(np.array(St_v2) - np.array(St_fom))
err_St_v1 = np.abs(np.array(St_v1) - np.array(St_fom))

print('=' * 60)
print(f'  Re={cfg.Re},  D={cfg.D},  M_train={M_TRAIN},  M_test={M_TEST}')
print(f'  KROM: {cfg.Nx_krom}x{cfg.Ny_krom}={N_KROM} pts,  rho={RHO},  T={T_SNAP} steps')
print('=' * 60)
print(f'{"Metric":<32}{"KROM v2":>12}{"KROM v1":>12}')
print('-' * 56)
print(f'{"Mean final rel-L2 (phase-aligned)":<32}{err_v2[:,-1].mean():>12.4f}{err_v1[:,-1].mean():>12.4f}')
print(f'{"Std  final rel-L2":<32}{err_v2[:,-1].std():>12.4f}{err_v1[:,-1].std():>12.4f}')
print(f'{"Mean |Delta St|":<32}{err_St_v2.mean():>12.4f}{err_St_v1.mean():>12.4f}')
print(f'{"FOM mean Strouhal":<32}{np.mean(St_fom):>12.4f}{"":>12}')
print('=' * 60)
print(f'Reference St(Re=100) ≈ 0.17-0.20  (Williamson 1989)')